# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print("Dataset Name: ", metadata_obj.name)
print("Description: ", metadata_obj.description)
print("Published: ", getattr(metadata_obj, 'datePublished', None))
print("Identifier: ", getattr(metadata_obj, 'identifier', None))

## 2. Data Overview
Review available record sets, fields, and their IDs. All are referenced by their `@id` fields.

In [ ]:
# List all record sets
record_sets = list(dataset.metadata.record_sets) if hasattr(dataset.metadata, 'record_sets') else []

print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RECORD SET: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            f_id = field.get('@id', '(no id)')
            f_name = field.get('name', '(no name)')
            f_col = field.get('column', {}).get('@id', '(no column)') if 'column' in field else '(no column)'
            print(f"    - Field @id: {f_id}, Name: {f_name}, Column: {f_col}")
    print("")

# Show the @ids for user reference
print("RecordSet @id list:")
record_set_ids = [rs['@id'] for rs in record_sets]
print(record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

*If your dataset has only one main record set, we'll use that. Otherwise, you can select from the list printed above.*

In [ ]:
# Extract records and load as DataFrame for each record set
# If no record sets available, skip extraction
dataframes = {}
if len(record_set_ids) == 0:
    print("No record sets found in the Croissant schema.")
    dataframes = None
else:
    for record_set_id in record_set_ids:
        try:
            print(f"Loading record set: {record_set_id}")
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(f"  Number of rows: {len(df)}\n")
        except Exception as e:
            print(f"Failed to load {record_set_id}: {e}")

    # For demonstration, pick the first record set
    main_record_set_id = record_set_ids[0]
    print(f"Main record set ID for analysis: {main_record_set_id}\n")

    # Show the first few records
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by key attribute. All fields referenced by `@id`.

In [ ]:
import numpy as np

if dataframes is not None:
    df = dataframes[main_record_set_id]

    # Choose a numeric field for analysis
    # For demonstration, try to find a numeric column
    sample_numeric_field = None
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.number):
            sample_numeric_field = c
            break

    print(f"Analyzing numeric field: {sample_numeric_field}")

    if sample_numeric_field is not None:
        threshold = np.nanmean(df[sample_numeric_field])  # mean threshold
        filtered_df = df[df[sample_numeric_field] > threshold]
        print(f"Filtered records with {sample_numeric_field} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{sample_numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()) / filtered_df[sample_numeric_field].std()
        print(f"Normalized {sample_numeric_field} for filtered records:")
        print(filtered_df[[sample_numeric_field, norm_col]].head())

        # Try grouping by a categorical field
        group_field = None
        for c in df.columns:
            if df[c].dtype == object and c != sample_numeric_field:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field, dropna=True)[sample_numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {sample_numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found in this record set.")

## 5. Visualization
Visualize distributions or relationships between numeric and/or categorical fields in the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes is not None and sample_numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[sample_numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{sample_numeric_field}' in {main_record_set_id}")
    plt.xlabel(sample_numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=sample_numeric_field)
        plt.title(f"Boxplot of '{sample_numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) Croissant dataset on mass spectrometry analysis of extracellular vesicles from stimulated human mast cells, and demonstrated how to:
- Inspect dataset metadata and record sets by `@id`
- Load records into DataFrames for analysis
- Filter, normalize, and group on important fields
- Visualize distributions and relationships

You can now selectively analyze other record sets or fields using their `@id`, as demonstrated. With these steps, you can rapidly explore multi-table Croissant datasets for scientific and machine learning workflows.